## 4. Configuration de YOLOv9 (`WongKinYiu/yolov9`)

In [ ]:
from pathlib import Path
import sys, yaml, torch

YOLO_DIR = Path("../yolov9").resolve()
DATA_YAML = Path("../trafic_data/data_1.yaml").resolve()
CFG_YAML = YOLO_DIR / "models/detect/yolov9-c.yaml"
HYP_YAML = YOLO_DIR / "data/hyps/hyp.scratch-high.yaml"
WEIGHTS_PT = YOLO_DIR / "yolov9-c.pt"

for p in [YOLO_DIR, YOLO_DIR / "train_dual.py", CFG_YAML, HYP_YAML, WEIGHTS_PT, DATA_YAML]:
    print("[OK]" , p.exists(), "->", p.name)

sys.path.insert(0, str(YOLO_DIR))
from utils.general import check_dataset, intersect_dicts
from models.yolo import Model

data = check_dataset(str(DATA_YAML))
hyp = yaml.safe_load(HYP_YAML.read_text(encoding="utf-8"))
ckpt = torch.load(WEIGHTS_PT, map_location="cpu", weights_only=False)
model = Model(str(CFG_YAML), ch=3, nc=int(data["nc"]), anchors=hyp.get("anchors"))

matched = intersect_dicts(ckpt["model"].float().state_dict(), model.state_dict(), exclude=["anchor"])
model.load_state_dict(matched, strict=False)
print(f"Transfer Learning : {len(matched)}/{len(model.state_dict())} tenseurs chargés (nc={data['nc']})")

### Rôle des composants
- **Architecture (`yolov9-c.yaml`) :** structure du réseau (sans poids).
- **Poids (`yolov9-c.pt`) :** paramètres préentraînés sur COCO (Transfer Learning).
- **Hyperparamètres (`hyp.scratch-high.yaml`) :** réglages d'apprentissage et d'augmentation.
- **Script (`train_dual.py`) :** lance l'entraînement et sauvegarde `best.pt` / `last.pt`.